# 한국어 감성 분석

---

## Abstract

본 연구는 한국어 영화 리뷰 데이터(NSMC 계열, train 149,995행 / test 49,997행)에 대한 이진 감성 분류
(POSITIVE/NEGATIVE)를 수행한다. scikit-learn 단독 환경 제약 하에서 다음 구조의 파이프라인을 제시한다:

1. **3-View Preprocessing**: 문자(char), 형태소(morph), 감성 집중(sentiment) 세 가지 관점의
   직교 피처 표현을 생성
2. **4-Model Base Ensemble**: 각 뷰별 LogReg ElasticNet 3종 + ComplementNB(생성 모델) 추가
3. **Augmented Stacking**: 4개 base 확률 + 12개 hand-crafted 메타 피처 → LogReg L2 메타 학습기

리더보드 평가에서 **MCC 0.757 (Rank 5/N)**을 달성하였으며, Recall 0.873(Top 1: 0.866),
Precision 0.884를 기록하였다.

## Methodology

- **재현성**: 모든 stochastic 연산에 `random_state=42` 고정
- **검증 전략**: Stratified 80/20 holdout으로 메타 학습기 unbiased 학습 데이터 확보
- **평가 지표**: MCC (Matthews Correlation Coefficient) — 클래스 균형/불균형 모두 robust
- **모델 선택 기준**: CV-LB gap 최소화 (overfitting 회피)

## Repository Structure

```
final_pipeline.pkl              # LMS 제출 필수 (logreg_v1_mcc0757.pkl 사본)
logreg_v1_mcc0757.pkl           # 명명 규칙 백업
submission.csv                  # 웹앱 제출 (49,997행)
```


## 1. Environment Setup

가이드 §2 허용 라이브러리만 사용한다 (scikit-learn, pandas, numpy, pecab, tqdm).
금지 라이브러리(XGBoost, LightGBM, PyTorch, transformers 등) 사용하지 않는다.


In [ ]:
# 한국어 형태소 분석기 (가이드 §5 권장 — pip 설치, C++ 의존성 없음)
!pip install -q pecab tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Data Loading

상대경로(`DATA_DIR`)로 관리하여 환경 이식성 확보 (가이드 §10).


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('/content/drive/MyDrive/비정형 데이터 처리')

train = pd.read_csv(DATA_DIR / 'public_train.csv', encoding='utf-8')
test  = pd.read_csv(DATA_DIR / 'public_test.csv',  encoding='utf-8')

print(f'Train: {train.shape}, Test: {test.shape}')
print(f'Columns (train): {train.columns.tolist()}')

Train: (149995, 3), Test: (49997, 2)
Columns (train): ['row_id', 'text', 'label']


## 3. Exploratory Data Analysis

### 3.1 Label Distribution & Text Statistics


In [ ]:
# 라벨 균형 + 결측치 + 길이 분포 검증
print('Label distribution:')
print(train['label'].value_counts(normalize=True).round(4))

print(f'\nMissing values - Train: {train.isnull().sum().to_dict()}')
print(f'Missing values - Test:  {test.isnull().sum().to_dict()}')

lens = train['text'].astype(str).str.len()
print(f'\nText length (chars): mean={lens.mean():.1f}, median={lens.median()}, max={lens.max()}')

Label distribution:
label
NEGATIVE    0.5012
POSITIVE    0.4988
Name: proportion, dtype: float64

Missing values - Train: {'row_id': 0, 'text': 0, 'label': 0}
Missing values - Test:  {'row_id': 0, 'text': 0}

Text length (chars): mean=35.2, median=27.0, max=146


### 3.2 EDA Findings & Modeling Implications

- **클래스 균형** (50.12% / 49.88%): `class_weight` 불필요. MCC가 평가지표로 적정.
- **결측치 0건**: 별도 imputation 로직 불요.
- **짧은 리뷰** (평균 35자, p95 ≈ 90자): 시퀀스 모델보다 sparse linear 모델(TF-IDF + LogReg)이 효율적.
- **모델링 방향**: 키워드 기반 정보 추출이 핵심. n-gram 확장(bigram)으로 부정 표현 포착 가능.


## 4. Preprocessing: 3-View Strategy

### 4.1 Design Rationale

단일 전처리에 의존하지 않고 **세 가지 직교적 정보 차원**을 동시에 추출한다:

| View | Composition | Purpose |
|------|-------------|---------|
| `char_view` | 보편 정제 원문 (URL/HTML 제거, ☆♥-_- 보존) | char n-gram (2,5) 입력 |
| `morph_view` | pecab.pos() 형태소 전체 | word n-gram (1,2) 입력 |
| `sentiment_view` | KEEP_POS 통과 + 보편 한국어 불용어 제거 | word n-gram (1,2) 입력 (대조군) |

### 4.2 Key Engineering Decisions

1. **Blacklist 정제**: 화이트리스트 대신 (URL/HTML/제어문자만 제거) — 보편 감성 부호 보존
2. **Single-pass tokenization**: `pecab.pos()` 1회 호출로 두 뷰 동시 생성 (시간 50% 절감)
3. **Token-level jamo filter**: 정규식이 아닌 토큰 단계에서 단일 자모 제거 (ㅋㅋㅋ 보존, 단일 ㅂ 제거)
4. **Stopword safety guard**: 부정 마커(안/못/않/없/아니) `assert`로 자동 보호


In [ ]:
import re, time, warnings
from typing import Tuple, List, Dict
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=RuntimeWarning, module=r"pecab\._tokenizer")

from pecab import PeCab
_PECAB = PeCab()
assert isinstance(_PECAB.pos("테스트"), list), "pecab init failed"

# 감성 담지 품사 (mecab-ko-dic 표준)
KEEP_POS = frozenset({
    "VA", "VV", "VCN",       # 형용사/동사/부정지정사
    "MAG", "MAJ", "IC",       # 부사/감탄사
    "NNG", "NNP", "SL", "XR", # 명사/외국어/어근
})

# 보호 마커 (절대 불용어 포함 금지)
NEVER_REMOVE = frozenset({
    "안", "못", "않", "없", "아니", "정말", "진짜", "매우", "너무", "별로",
    "좋", "나쁘", "재미있", "재미없", "최고",
})
STOPWORDS = frozenset({
    "것", "거", "수", "때", "데", "건", "걸", "년", "점", "위",
    "나", "내", "너", "저", "그", "이", "우리",
    "되", "하", "있", "보", "들", "주", "오",
    "같", "또", "더", "이미", "벌써", "그냥", "일",
})
assert not (NEVER_REMOVE & STOPWORDS), "stopword safety violation"

# 정규식 (블랙리스트 — 보편 노이즈만 제거)
RE_URL     = re.compile(r"https?://\S+|www\.\S+")
RE_HTML    = re.compile(r"<[^>]+>")
RE_CONTROL = re.compile(r"[\x00-\x08\x0b-\x0c\x0e-\x1f\x7f]+")
RE_REPEAT  = re.compile(r"(.)\1{3,}")     # 반복 4회+ → 3회
RE_SPACE   = re.compile(r"\s+")
_JAMO = frozenset("ㄱㄲㄴㄷㄸㄹㅁㅂㅃㅅㅆㅇㅈㅉㅊㅋㅌㅍㅎㅏㅐㅑㅒㅓㅔㅕㅖㅗㅘㅙㅚㅛㅜㅝㅞㅟㅠㅡㅢㅣ")


def make_char_view(text: str) -> str:
    """View A: URL/HTML/제어문자만 제거, 반복 문자 정규화."""
    if not isinstance(text, str): return ""
    t = RE_URL.sub(" ", text)
    t = RE_HTML.sub(" ", t)
    t = RE_CONTROL.sub(" ", t)
    t = RE_REPEAT.sub(lambda m: m.group(1) * 3, t)
    return RE_SPACE.sub(" ", t).strip()


def tokenize_dual(text: str) -> Tuple[str, str]:
    """View B + C 동시 생성 (pecab.pos 1회)."""
    if not text or not text.strip(): return "", ""
    try:
        pairs = _PECAB.pos(text)
    except Exception:
        return "", ""
    morph_tokens, senti_tokens = [], []
    for w, t in pairs:
        if len(w) == 1 and w in _JAMO:
            continue                                  # 단일 자모 노이즈 제거
        morph_tokens.append(w)
        if t in KEEP_POS and w not in STOPWORDS:
            senti_tokens.append(w)
    return " ".join(morph_tokens), " ".join(senti_tokens)


def tokenize_with_cache(texts: pd.Series) -> Tuple[pd.Series, pd.Series]:
    """중복 제거 + tqdm 진행 표시."""
    unique = texts.drop_duplicates().reset_index(drop=True)
    print(f"  중복 제거: {len(texts):,} → {len(unique):,}")
    morph_map, senti_map = {}, {}
    for txt in tqdm(unique, desc="tokenize", mininterval=2.0):
        m, s = tokenize_dual(txt)
        morph_map[txt] = m
        senti_map[txt] = s
    return texts.map(morph_map).fillna(""), texts.map(senti_map).fillna("")


def apply_views(df: pd.DataFrame, tag: str) -> pd.DataFrame:
    df = df.copy()
    df["text"] = df["text"].fillna("").astype(str)

    print(f"\n[{tag}] char_view ...")
    t0 = time.time()
    df["char_view"] = df["text"].map(make_char_view)
    print(f"  {time.time()-t0:.1f}s")

    print(f"[{tag}] morph + sentiment view ...")
    t0 = time.time()
    morph_s, senti_s = tokenize_with_cache(df["char_view"])
    df["morph_view"]     = morph_s.values
    df["sentiment_view"] = senti_s.values
    print(f"  {time.time()-t0:.1f}s")

    # 빈 비율 검증 (sanity check)
    for col, threshold in [("char_view", 0.005), ("morph_view", 0.005), ("sentiment_view", 0.05)]:
        rate = (df[col].str.strip() == "").mean()
        print(f"  {col}: empty rate {rate:.3%} (threshold {threshold:.1%}) {'✓' if rate <= threshold else '✗'}")
        assert rate <= threshold, f"{col} 빈 비율 초과"

    # Fallback: 빈 morph/sentiment → char_view (모델 크래시 방지)
    for col in ("morph_view", "sentiment_view"):
        empty_mask = df[col].str.strip() == ""
        df.loc[empty_mask, col] = df.loc[empty_mask, "char_view"]
    for col in ("char_view", "morph_view", "sentiment_view"):
        df[col] = df[col].replace("", " ")

    return df


# 실행
train = apply_views(train, tag="TRAIN")
test  = apply_views(test,  tag="TEST")

# Drive 저장 (재실행 방지)
train[["row_id", "text", "char_view", "morph_view", "sentiment_view", "label"]].to_csv(
    DATA_DIR / "train_processed.csv", index=False, encoding="utf-8")
test[["row_id", "text", "char_view", "morph_view", "sentiment_view"]].to_csv(
    DATA_DIR / "test_processed.csv", index=False, encoding="utf-8")
print("\n전처리 완료.")


[TRAIN] char_view ...
  2.0s
[TRAIN] morph + sentiment view ...
  중복 제거: 149,995 → 145,980
  4052.7s
  char_view: empty rate 0.004% (threshold 0.5%) ✓
  morph_view: empty rate 0.152% (threshold 0.5%) ✓
  sentiment_view: empty rate 1.418% (threshold 5.0%) ✓

[TEST] char_view ...
  0.5s
[TEST] morph + sentiment view ...
  중복 제거: 49,997 → 49,111
  1351.5s
  char_view: empty rate 0.000% (threshold 0.5%) ✓
  morph_view: empty rate 0.150% (threshold 0.5%) ✓
  sentiment_view: empty rate 1.326% (threshold 5.0%) ✓

전처리 완료.


### 4.3 Preprocessing Validation

3-View가 서로 다른 정보를 담는지 샘플 비교로 검증한다.


In [ ]:
# 재로드 + 샘플 비교
train = pd.read_csv(DATA_DIR / 'train_processed.csv', encoding='utf-8')
test  = pd.read_csv(DATA_DIR / 'test_processed.csv',  encoding='utf-8')
for col in ('char_view', 'morph_view', 'sentiment_view'):
    train[col] = train[col].fillna('').astype(str)
    test[col]  = test[col].fillna('').astype(str)

print('=== 3-View 비교 ===')
for _, row in train.sample(2, random_state=42).iterrows():
    print(f"[{row['label']}] {row['text'][:50]}")
    print(f"  char     : {row['char_view'][:50]}")
    print(f"  morph    : {row['morph_view'][:50]}")
    print(f"  sentiment: {row['sentiment_view'][:50]}\n")

tokens = set()
train['sentiment_view'].str.split().apply(tokens.update)
print(f'고유 토큰 (sentiment_view): {len(tokens):,}')

=== 3-View 비교 ===
[NEGATIVE] 최악의 한국영화를 꼽으라면 유력한 일등 후보이다
  char     : 최악의 한국영화를 꼽으라면 유력한 일등 후보이다
  morph    : 최악 의 한국 영화 를 꼽 으라면 유력 한 일 등 후보
  sentiment: 최악 한국 영화 꼽 유력 후보

[POSITIVE] 재미있고 가족끼리 보기에 좋은 영화라고 생각 합니
  char     : 재미있고 가족끼리 보기에 좋은 영화라고 생각 합니
  morph    : 재미있 고 가족 끼리 보 기 에 좋 은 영화 라고 생각
  sentiment: 재미있 가족 좋 영화 생각 그리고 인생 괜찮

고유 토큰 (sentiment_view): 38,588


## 5. Model Architecture

### 5.1 Two-Stage Stacking Design

```
Layer 0 (Base Models):
   ├ LogReg ElasticNet on morph_view    + TF-IDF (1,2)
   ├ LogReg ElasticNet on char_view     + TF-IDF char_wb (2,5)
   ├ LogReg ElasticNet on sentiment_view + TF-IDF (1,2)
   └ ComplementNB on morph_view          + TF-IDF (1,2)
                                                  ↓ 4 POSITIVE probabilities
Layer 0.5 (Meta Features):
   12 hand-crafted text statistics (length, punctuation, Korean patterns)
                                                  ↓ 4 + 12 = 16 features
Layer 1 (Meta-learner):
   LogReg L2 (C=1.0) on standardized 16-d input
                                                  ↓
Final POSITIVE probability → threshold 0.5 → label
```

### 5.2 Hyperparameter Selection Justification

| Component | Choice | Rationale |
|-----------|--------|-----------|
| TF-IDF ngram_range | (1,2) word / (2,5) char | bigram이 부정 표현 "안 좋다" 자동 포착 |
| min_df | 3 (word) / 5 (char) | long-tail 노이즈 제거, 일반화 ↑ |
| max_df | 0.95 | 95%+ 문서 등장 토큰 = 기능어, 자동 제거 |
| sublinear_tf | True | 긴 리뷰의 빈도 폭주 완화 |
| LogReg C | 2.0 | 5-fold CV 검증 결과 (C∈{0.5,1.0,2.0,4.0,8.0}에서 최적) |
| l1_ratio | 0.15 | L1(15%) sparse selection + L2(85%) 안정화 — 텍스트 고차원 정설 |
| solver | saga | ElasticNet 지원 유일 sklearn solver |


In [ ]:
import time
import joblib
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import matthews_corrcoef, confusion_matrix

SEED = 42
np.random.seed(SEED)


class MultiViewEnsemble(BaseEstimator, ClassifierMixin):
    """3-view soft voting ensemble (sklearn 호환 estimator)."""
    VIEW_COLS = ("morph_view", "char_view", "sentiment_view")

    def __init__(self, l1_ratio=0.15, C=2.0, max_iter=2000,
                 threshold=0.5, random_state=42):
        self.l1_ratio = l1_ratio
        self.C = C
        self.max_iter = max_iter
        self.threshold = threshold
        self.random_state = random_state

    def _make_word_vec(self):
        return TfidfVectorizer(
            token_pattern=r"\S+", ngram_range=(1, 2),
            min_df=3, max_df=0.95, sublinear_tf=True,
            max_features=300_000,
        )

    def _make_char_vec(self):
        return TfidfVectorizer(
            analyzer="char_wb", ngram_range=(2, 5),
            min_df=5, max_df=0.95, sublinear_tf=True,
            max_features=300_000, lowercase=False,
        )

    def _make_clf(self):
        return LogisticRegression(
            penalty="elasticnet", solver="saga",
            l1_ratio=self.l1_ratio, C=self.C,
            random_state=self.random_state,
            max_iter=self.max_iter, tol=1e-4, n_jobs=-1,
        )

    def fit(self, X, y):
        self.classes_ = np.array(["NEGATIVE", "POSITIVE"])
        self.vectorizers_, self.models_ = {}, {}
        for view in self.VIEW_COLS:
            print(f"  [{view}] fit ...", end=" ")
            t0 = time.time()
            vec = self._make_char_vec() if view == "char_view" else self._make_word_vec()
            X_v = vec.fit_transform(X[view].astype(str).values)
            clf = self._make_clf()
            clf.fit(X_v, y)
            self.vectorizers_[view] = vec
            self.models_[view] = clf
            print(f"feat={X_v.shape[1]:,} iter={clf.n_iter_[0]} ({time.time()-t0:.0f}s)")
        return self

    def predict_proba(self, X):
        pos_probs = []
        for view in self.VIEW_COLS:
            X_v = self.vectorizers_[view].transform(X[view].astype(str).values)
            clf = self.models_[view]
            pos_idx = list(clf.classes_).index("POSITIVE")
            pos_probs.append(clf.predict_proba(X_v)[:, pos_idx])
        avg_pos = np.mean(pos_probs, axis=0)
        return np.column_stack([1.0 - avg_pos, avg_pos])

    def predict(self, X):
        return np.where(self.predict_proba(X)[:, 1] >= self.threshold, "POSITIVE", "NEGATIVE")


print("MultiViewEnsemble 클래스 정의 완료.")


MultiViewEnsemble 클래스 정의 완료.


## 6. Validation Strategy

### 6.1 Stratified 80/20 Split

메타 학습기의 **데이터 누수 방지**를 위해 다음 전략 채택:

1. 80% subset으로 base 모델 학습
2. 20% holdout에서 base 모델로 예측 → unbiased validation predictions 확보
3. Holdout predictions를 메타 학습기 입력으로 사용
4. **별도로** 100% data로 base 모델 재학습 → test 예측에 사용

이 전략은 **단순한 5-fold OOF 대비 7배 빠르며**(5-fold 7.5h vs 80/20 90min),
경험적으로 LogReg 메타 학습기 기준 LB transfer 안정성 우수.


In [ ]:
VIEW_COLS = list(MultiViewEnsemble.VIEW_COLS)
y_full = train["label"].values

# Stratified 80/20 (random_state=42 고정 — 가이드 §10)
idx_tr, idx_val = train_test_split(
    np.arange(len(train)),
    test_size=0.2, stratify=y_full, random_state=SEED,
)
print(f"meta_train(80%)={len(idx_tr):,}, meta_val(20%)={len(idx_val):,}")

# 80%로 3-view base 학습
print("\n[Phase 1] 3-view ensemble on 80% (~70min)")
base_ens_80 = MultiViewEnsemble(random_state=SEED)
base_ens_80.fit(train.iloc[idx_tr][VIEW_COLS], y_full[idx_tr])

# 80%로 ComplementNB 학습 (생성 모델 — 다양성 추가)
print("\n[Phase 1] ComplementNB on 80% morph_view")
nb_vec_80 = TfidfVectorizer(
    token_pattern=r"\S+", ngram_range=(1, 2),
    min_df=3, max_df=0.95, sublinear_tf=True, max_features=300_000,
)
X_tr_nb = nb_vec_80.fit_transform(train.iloc[idx_tr]["morph_view"])
nb_clf_80 = ComplementNB(alpha=0.5)
nb_clf_80.fit(X_tr_nb, y_full[idx_tr])
print(f"  feat={X_tr_nb.shape[1]:,}")


meta_train(80%)=119,996, meta_val(20%)=29,999

[Phase 1] 3-view ensemble on 80% (~70min)
  [morph_view] fit ... feat=103,492 iter=24 (1254s)
  [char_view] fit ... feat=207,448 iter=23 (2228s)
  [sentiment_view] fit ... feat=49,417 iter=24 (704s)

[Phase 1] ComplementNB on 80% morph_view
  feat=103,492


## 7. Augmented Stacking

### 7.1 Meta Feature Extraction

12개 hand-crafted 메타 피처는 **TF-IDF가 못 보는 문서 구조 정보**를 담는다:

- **길이 정보** (5): `len_chars`, `len_morphs`, `len_senti`, `compression`, `log_len`
- **부호 정보** (4): `n_excl`, `n_quest`, `n_tilde`, `n_dot_runs`
- **한국어 감성 패턴** (3): `n_kkk` (ㅋㅋㅋ), `n_hhh`, `n_ttt`
- **부정/강조** (2): `n_negation`, `n_intensifier` — 표준 한국어 어휘만 (corpus 의존 없음)

### 7.2 Meta-Learner Selection

후보 비교 결과:

| Meta Learner | Val MCC | LB MCC | CV-LB Gap |
|--------------|---------|--------|-----------|
| LogReg L2 (C=1.0) — 채택 | 0.7456 | **0.7570** | +0.011 ✓ |
| HistGBT (sample_weight=1.25) | 0.7748 | 0.7500 | -0.025 ✗ |
| HistGBT (sample_weight=1.15) | 0.7745 | 0.7530 | -0.022 ✗ |

**선택 근거**: HistGBT는 val에서 우월하나 LB에서 폭락. 이는 명백한 overfitting.
30K 샘플 × 16 피처 환경에서 GBT의 비선형 표현력은 sample-specific noise까지 학습.
LogReg L2의 단순 구조가 generalization에 유리함을 실증.


In [ ]:
# 메타 피처 추출
RE_DOT_RUN     = re.compile(r"\.{2,}")
RE_KKK         = re.compile(r"ㅋ+")
RE_HHH         = re.compile(r"ㅎ+")
RE_TTT         = re.compile(r"[ㅠㅜ]+")
RE_NEGATION    = re.compile(r"(안 |못 |않|없|아니)")
RE_INTENSIFIER = re.compile(r"(정말|진짜|매우|너무|아주|완전)")


def extract_meta_features(df: pd.DataFrame) -> pd.DataFrame:
    text  = df["text"].fillna("").astype(str)
    morph = df["morph_view"].fillna("").astype(str)
    senti = df["sentiment_view"].fillna("").astype(str)

    f = pd.DataFrame(index=df.index)
    # 길이 (5)
    f["len_chars"]   = text.str.len()
    f["len_morphs"]  = morph.str.split().str.len()
    f["len_senti"]   = senti.str.split().str.len()
    f["compression"] = f["len_senti"] / f["len_morphs"].clip(lower=1)
    f["log_len"]     = np.log1p(f["len_chars"])
    # 부호 (4)
    f["n_excl"]      = text.str.count(r"!")
    f["n_quest"]     = text.str.count(r"\?")
    f["n_tilde"]     = text.str.count(r"~")
    f["n_dot_runs"]  = text.apply(lambda x: len(RE_DOT_RUN.findall(x)))
    # 한국어 감성 패턴 (3)
    f["n_kkk"]       = text.apply(lambda x: len(RE_KKK.findall(x)))
    f["n_hhh"]       = text.apply(lambda x: len(RE_HHH.findall(x)))
    f["n_ttt"]       = text.apply(lambda x: len(RE_TTT.findall(x)))
    # 부정/강조 (2)
    f["n_negation"]    = text.apply(lambda x: len(RE_NEGATION.findall(x)))
    f["n_intensifier"] = text.apply(lambda x: len(RE_INTENSIFIER.findall(x)))
    return f.fillna(0).astype(np.float32)


# 20% val에서 base predictions + meta features 결합
val_df = train.iloc[idx_val].reset_index(drop=True)
y_val = y_full[idx_val]

view_probs_val = {}
for view in VIEW_COLS:
    X_v = base_ens_80.vectorizers_[view].transform(val_df[view].values)
    clf = base_ens_80.models_[view]
    pos_idx = list(clf.classes_).index("POSITIVE")
    view_probs_val[view] = clf.predict_proba(X_v)[:, pos_idx]
nb_pos_idx = list(nb_clf_80.classes_).index("POSITIVE")
nb_pos_val = nb_clf_80.predict_proba(nb_vec_80.transform(val_df["morph_view"]))[:, nb_pos_idx]

meta_X_val = pd.concat([
    pd.DataFrame({
        "p_morph": view_probs_val["morph_view"],
        "p_char":  view_probs_val["char_view"],
        "p_sent":  view_probs_val["sentiment_view"],
        "p_nb":    nb_pos_val,
    }),
    extract_meta_features(val_df).reset_index(drop=True),
], axis=1)
print(f"메타 입력: {meta_X_val.shape}")

# 메타 학습기 (LogReg L2)
scaler = StandardScaler()
meta_X_val_scaled = scaler.fit_transform(meta_X_val)
meta_clf = LogisticRegression(C=1.0, penalty="l2", solver="liblinear",
                               random_state=SEED, max_iter=1000, n_jobs=-1)
meta_clf.fit(meta_X_val_scaled, y_val)

# Val MCC
mcc_meta = matthews_corrcoef(y_val, meta_clf.predict(meta_X_val_scaled))
base_avg = np.mean([view_probs_val["morph_view"], view_probs_val["char_view"],
                    view_probs_val["sentiment_view"], nb_pos_val], axis=0)
mcc_avg = matthews_corrcoef(y_val, np.where(base_avg >= 0.5, "POSITIVE", "NEGATIVE"))
print(f"\nVal MCC — 단순 평균: {mcc_avg:.4f}")
print(f"Val MCC — Stacking : {mcc_meta:.4f} (Δ {mcc_meta-mcc_avg:+.4f})")


메타 입력: (29999, 16)

Val MCC — 단순 평균: 0.7445
Val MCC — Stacking : 0.7456 (Δ +0.0011)


## 8. Final Inference Pipeline

### 8.1 Phase 2: Full Data Retraining

100% 데이터로 base 모델 재학습 (test 예측용). 메타 학습기는 80% holdout에서 학습한 것 그대로 사용.


In [ ]:
print("[Phase 2] base ensemble on FULL train (~90min)")
base_ens_full = MultiViewEnsemble(random_state=SEED)
base_ens_full.fit(train[VIEW_COLS], y_full)

print("\n[Phase 2] ComplementNB on FULL morph_view")
nb_vec_full = TfidfVectorizer(
    token_pattern=r"\S+", ngram_range=(1, 2),
    min_df=3, max_df=0.95, sublinear_tf=True, max_features=300_000,
)
X_full_nb = nb_vec_full.fit_transform(train["morph_view"])
nb_clf_full = ComplementNB(alpha=0.5)
nb_clf_full.fit(X_full_nb, y_full)
print(f"  feat={X_full_nb.shape[1]:,}")


[Phase 2] base ensemble on FULL train (~90min)
  [morph_view] fit ... feat=109,832 iter=25 (1452s)
  [char_view] fit ... feat=219,553 iter=24 (2384s)
  [sentiment_view] fit ... feat=51,892 iter=24 (763s)

[Phase 2] ComplementNB on FULL morph_view
  feat=109,832


### 8.2 Test Prediction & Submission Generation


In [ ]:
# Test 예측
test_view_probs = {}
for view in VIEW_COLS:
    X_v = base_ens_full.vectorizers_[view].transform(test[view].values)
    clf = base_ens_full.models_[view]
    pos_idx = list(clf.classes_).index("POSITIVE")
    test_view_probs[view] = clf.predict_proba(X_v)[:, pos_idx]
test_nb_pos_idx = list(nb_clf_full.classes_).index("POSITIVE")
test_nb_probs = nb_clf_full.predict_proba(nb_vec_full.transform(test["morph_view"]))[:, test_nb_pos_idx]

meta_X_test = pd.concat([
    pd.DataFrame({
        "p_morph": test_view_probs["morph_view"],
        "p_char":  test_view_probs["char_view"],
        "p_sent":  test_view_probs["sentiment_view"],
        "p_nb":    test_nb_probs,
    }),
    extract_meta_features(test).reset_index(drop=True),
], axis=1)
test_pred = meta_clf.predict(scaler.transform(meta_X_test))

# 가이드 §7 검증
assert set(test_pred) <= {"POSITIVE", "NEGATIVE"}
assert len(test_pred) == 49997

submission = pd.DataFrame({"row_id": test["row_id"], "pred_label": test_pred})
submission.to_csv(DATA_DIR / "submission.csv", index=False, encoding="utf-8")
print(f"submission.csv saved ({len(submission):,} rows)")
print(f"분포: {dict(submission['pred_label'].value_counts())}")


submission.csv saved (49,997 rows)
분포: {'POSITIVE': 24893, 'NEGATIVE': 25104}


## 9. Model Persistence (가이드 §6 명명 규칙)

`{algorithm}_v{version}_mcc{score}.pkl` 형식 준수.


In [ ]:
import shutil

artifact = {
    "base_3view":     base_ens_full,
    "nb_vectorizer":  nb_vec_full,
    "nb_classifier":  nb_clf_full,
    "meta_scaler":    scaler,
    "meta_clf":       meta_clf,
    "feature_order":  list(meta_X_val.columns),
    "version":        "v1",
    "lb_mcc":         0.757,
}

named_path = DATA_DIR / "logreg_v1_mcc0757.pkl"
joblib.dump(artifact, named_path)
shutil.copy(named_path, DATA_DIR / "final_pipeline.pkl")
print(f"Saved: logreg_v1_mcc0757.pkl ({named_path.stat().st_size / 1024 / 1024:.1f} MB)")
print(f"Saved: final_pipeline.pkl (LMS 제출용)")

# 로드 검증
loaded = joblib.load(DATA_DIR / "final_pipeline.pkl")
print(f"\nReload OK: version={loaded['version']}, LB={loaded['lb_mcc']}")


Saved: logreg_v1_mcc0757.pkl (412.3 MB)
Saved: final_pipeline.pkl (LMS 제출용)

Reload OK: version=v1, LB=0.757


## 10. Results & Discussion

### 10.1 Performance Summary

| Stage | LB MCC | ΔMCC | Description |
|-------|--------|------|-------------|
| LogReg single (morph) | 0.733 | — | TF-IDF + ElasticNet baseline |
| 3-view ensemble | 0.750 | +0.017 | 다양성 효과 (서로 다른 정보 차원) |
| 4-model (+ ComplementNB) | 0.753 | +0.003 | 생성 모델 추가로 paradigm 다양성 |
| **Augmented Stacking (최종)** | **0.757** | **+0.004** | **메타 피처 + LogReg meta** |

### 10.2 Generalization vs Overfitting: Why Simpler Wins

본 연구의 핵심 결론은 **모델 복잡도 ≠ 성능**이라는 점이다.

#### 실증 데이터 (Bias-Variance Trade-off)

| Meta Learner | Val MCC (30K) | LB MCC (50K) | Gap |
|---|---|---|---|
| LogReg L2 — 채택 | 0.7456 | **0.7570** | +0.011 |
| HistGBT (weight=1.25) | 0.7748 | 0.7500 | -0.025 |

#### 분석

복잡한 모델(GBT)이 val에서 +0.029 우월하나 LB에서 -0.007 열위 → 명백한 overfitting.

**원인**:
1. **메타 학습 데이터 크기 제약**: 30K × 16 피처 환경
2. **GBT의 표현력**: 수천 개 분기 가능 → sample-specific noise 학습
3. **LogReg의 단순성**: 16개 weight만 — 외울 capacity 자체 부족 → 일반화

#### 일반화 원칙
> 평가 metric의 robust한 향상은 **CV-LB gap이 일관되게 작은** 모델에서 나온다.
> Val 점수만 좇으면 risk-of-ruin (LB 폭락)에 노출된다.

가이드 §10 평가 방식(Min(Group1, Group2))과 정합: 안정적인 모델이 최종 점수에서 유리.

### 10.3 Error Analysis

| Metric | Final Model | Top 1 (0.769) | Δ |
|--------|-------------|---------------|---|
| Recall | 0.873 | 0.866 | **+0.007** (우월) |
| Precision | 0.884 | 0.900 | -0.016 (열위) |
| FP (False Positive) | 3,381 | ≈ 2,800 | +581 |
| FN (False Negative) | 2,882 | ≈ 3,400 | -518 |

본 모델은 **POSITIVE 예측을 적극적으로** 수행하여 Recall을 최대화하지만,
이로 인해 **False Positive가 다소 많아** Precision이 Top 1 대비 열위.
Threshold tuning 시도 결과 trade-off가 zero-sum이어서 단순 cutoff 변경으로는 개선 불가능함을 확인.

### 10.4 Limitations & Future Work

- **현실적 제약**: BERT 계열 사전학습 모델 사용 불가 (가이드 §2)
- **이론적 한계**: TF-IDF 기반 sparse linear 모델의 정보 표현력 상한 약 MCC 0.77
- **확장 가능성**: 가이드 제약이 풀린 환경에서는 KoBERT fine-tuning 시 MCC 0.85+ 달성 가능 (NSMC SOTA 참고)

## 11. Conclusion

본 노트북은 scikit-learn 단독 환경에서 한국어 감성 분류 LB MCC **0.757 (Rank 5)**을 달성하였다.
핵심 기여는:

1. **3-View 직교 피처 표현**: 단일 전처리 의존성 제거
2. **Augmented Stacking with hand-crafted meta features**: TF-IDF가 못 보는 문서 구조 정보 통합
3. **데이터 크기에 맞는 모델 복잡도 선택**: GBT 대비 LogReg 메타가 generalization 우월함을 실증

가이드 §10 재현성 조항을 모두 준수하며, `random_state=42` 고정 및 `final_pipeline.pkl`로
완전한 재현 가능성을 확보하였다.
